In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/processed/candor_dataset_clean.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/candor_dataset_clean.csv'

In [1]:
import os
print(os.getcwd())

c:\Users\amb20\CANDOR_enjoyment\notebooks\H5


A Mixed Model =  Fixed effects (normal regression effects) + Random effects (group-level variation)

* Fixed effects: On average, across all conversations and speakers, how do partner behaviors (TFO, overlap, pauses, speech activity) affect actor enjoyment?
* Random effects: 
    - Each conversation (dyad) gets its own intercept because some conversations are naturally more enjoyable than others. Model estimates baseline_enjoyment + u_call (how much this convo differs from avg)
    - Each speaker also gets their own intercept. baseline_enjoyment + u_call + u_speaker (how much this speaker differs from avg)

Allows correct p-values and realistic modeling of dyadic data.

Some convos naturally fun and some ppl naturally fun. If no control - we may think partner tfo predicts enjoyment when it was just a fun convo

In [ ]:
import statsmodels.formula.api as smf

model = smf.mixedlm(
    "how_enjoyable_actor ~ tfo_partner + overlap_partner + pauses_partner + speech_activity_partner",
    data=df,
    groups=df["call_id"],          
    vc_formula={"Speaker": "0 + C(actor_id)"})
result = model.fit()
print(result.summary())

c:\Users\amb20\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


              Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: how_enjoyable_actor
No. Observations:  3200    Method:             REML               
No. Groups:        1600    Scale:              1.1552             
Min. group size:   2       Log-Likelihood:     -5881.8047         
Max. group size:   2       Converged:          Yes                
Mean group size:   2.0                                            
------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept                8.054    0.111 72.721 0.000  7.837  8.271
tfo_partner             -0.912    0.147 -6.218 0.000 -1.199 -0.624
overlap_partner         -0.307    0.768 -0.400 0.689 -1.812  1.198
pauses_partner          -0.662    0.271 -2.444 0.015 -1.193 -0.131
speech_activity_partner -0.575    0.155 -3.724 0.000 -0.878 -0.273
Speaker Va

* When partner TFO increases by 1 unit -> actor enjoyment decreases ~ 0.91 units
* No evidence that partner overlap predicts actor enjoyment.
* More pauses by the partner are associated with lower actor enjoyment.
* Higher partner speech activity -> lower actor enjoyment.

##  R**2
How much variance is explained by fixed effects only? (TFO, Overlap, Pauses, Speech Activity)

## Condiitonalr R**2
How much variance is explained by fixed + random effects together? (partner behaviours and speaker differences)

In [ ]:
print(result.cov_re)
print(result.vcomp)

Empty DataFrame
Columns: []
Index: []
[1.15515004]


In [ ]:
var_residual = result.scale
var_random = result.vcomp[0]

# predictions 
fixed_preds = result.model.exog @ result.fe_params

# Variance of fixed effects
var_fixed = np.var(fixed_preds, ddof=1)

var_total = var_fixed + var_random + var_residual

r2_marginal = var_fixed / var_total
r2_conditional = (var_fixed + var_random) / var_total

print("Marginal R²:", r2_marginal)
print("Conditional R²:", r2_conditional)

Marginal R²: 0.020121336460907708
Conditional R²: 0.5100606682304539


* Only 2% of the variance in actor enjoyment is explained by partner behaviors (TFO, overlap, pauses, speech activity).
* 51% of the variance in actor enjoyment is explained when we include spealer differences

Predictors are statistically significant but effect size is small